In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
import ast
import plotly.graph_objects as go

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

In [2]:
sensor_df = pd.read_parquet("sensor.parquet")
event_df = pd.read_parquet("event.parquet")

print(sensor_df.shape, event_df.shape)

(10000, 7) (200, 14)


In [3]:
def safe_parse_array(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.array(x, dtype=np.float32)

    if isinstance(x, str):
        try:
            return np.array(ast.literal_eval(x), dtype=np.float32)
        except:
            x = x.replace("[", "").replace("]", "")
            return np.fromstring(x, sep=" ", dtype=np.float32)

    return np.zeros(0, dtype=np.float32)

In [4]:
LAT_MEAN = sensor_df["latitude"].mean()
LON_MEAN = sensor_df["longitude"].mean()

LAT_STD = sensor_df["latitude"].std() + 1e-6
LON_STD = sensor_df["longitude"].std() + 1e-6

def normalize_xy(xy):
    xy[:, 0] = (xy[:, 0] - LAT_MEAN) / LAT_STD
    xy[:, 1] = (xy[:, 1] - LON_MEAN) / LON_STD
    return xy

In [5]:
def pad_or_truncate(seq, L=10):
    if len(seq) >= L:
        return seq[:L]
    else:
        pad = np.repeat(seq[-1:], L - len(seq), axis=0)
        return np.vstack([seq, pad])

In [6]:
def build_subtracks(sensor_df, L=10):
    sub_tracks = []

    for tid, g in sensor_df.groupby("trackId"):
        g = g.sort_values("timestamp")
        xy = g[["latitude", "longitude"]].values

        for w in range(5, 11):
            for i in range(0, len(xy) - w + 1, 2):
                seg = xy[i:i+w].copy()

                seg = normalize_xy(seg)
                seg = pad_or_truncate(seg, L)   # 🔥 FIX

                sub_tracks.append({
                    "track_id": tid,
                    "segment": seg
                })

    return sub_tracks

sub_tracks = build_subtracks(sensor_df)
print("Subtracks:", len(sub_tracks))

Subtracks: 21000


In [7]:
def encode_report_seq(row, L=10):
    lons = safe_parse_array(row["coors.longitudes"])
    lats = safe_parse_array(row["coors.latitudes"])

    n = min(len(lons), len(lats))
    if n == 0:
        return np.zeros((L, 2), dtype=np.float32)

    xy = np.stack([lats[:n], lons[:n]], axis=1)

    # CASE 1: only 1 point → replicate
    if len(xy) == 1:
        xy = np.repeat(xy, L, axis=0)

    # CASE 2: pad if short
    if len(xy) < L:
        pad = np.repeat(xy[-1:], L - len(xy), axis=0)
        xy = np.vstack([xy, pad])

    # CASE 3: truncate
    xy = xy[:L]

    return normalize_xy(xy)

In [8]:
track_seqs = [s["segment"] for s in sub_tracks]
report_seqs = [encode_report_seq(r) for _, r in event_df.iterrows()]

In [9]:
class TrajTransformer(nn.Module):
    def __init__(self, d_model=64):
        super().__init__()

        self.input_proj = nn.Linear(2, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.encoder(x)
        return x.mean(dim=1)

In [10]:
class PairDataset(Dataset):
    def __init__(self, track_seqs, report_seqs, num_neg=3):
        self.data = []

        for rid, r in enumerate(report_seqs):
            pos = random.randint(0, len(track_seqs)-1)

            for _ in range(num_neg):
                neg = random.randint(0, len(track_seqs)-1)

                self.data.append((r, track_seqs[pos], track_seqs[neg]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        r, p, n = self.data[idx]
        return (
            torch.tensor(r),
            torch.tensor(p),
            torch.tensor(n)
        )

In [11]:
def spatial_loss(anchor, pos, neg, pos_xy, neg_xy, w_spatial=0.5):
    d_pos = F.pairwise_distance(anchor, pos)
    d_neg = F.pairwise_distance(anchor, neg)

    loss_embed = F.relu(d_pos - d_neg + 1.0)

    spatial_pos = ((pos_xy - pos_xy.mean(dim=1, keepdim=True))**2).sum()
    spatial_neg = ((neg_xy - neg_xy.mean(dim=1, keepdim=True))**2).sum()

    loss_spatial = spatial_pos - spatial_neg

    return (loss_embed + w_spatial * loss_spatial).mean()

In [12]:
net = TrajTransformer()

dataset = PairDataset(track_seqs, report_seqs)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

for ep in range(100):
    total = 0

    for r, p, n in loader:
        optimizer.zero_grad()

        e_r = net(r.float())
        e_p = net(p.float())
        e_n = net(n.float())

        loss = spatial_loss(e_r, e_p, e_n, p, n)
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {ep+1}: {total/len(loader):.4f}")

Epoch 1: 1.1400
Epoch 2: 0.9565
Epoch 3: 0.9646
Epoch 4: 0.9395
Epoch 5: 0.9382
Epoch 6: 0.9336
Epoch 7: 0.9401
Epoch 8: 0.9074
Epoch 9: 0.9343
Epoch 10: 0.9385
Epoch 11: 0.9204
Epoch 12: 0.9070
Epoch 13: 0.9133
Epoch 14: 0.8952
Epoch 15: 0.8898
Epoch 16: 0.9144
Epoch 17: 0.9165
Epoch 18: 0.8909
Epoch 19: 0.9068
Epoch 20: 0.8910
Epoch 21: 0.9193
Epoch 22: 0.8863
Epoch 23: 0.9085
Epoch 24: 0.8898
Epoch 25: 0.8836
Epoch 26: 0.8823
Epoch 27: 0.8849
Epoch 28: 0.8690
Epoch 29: 0.8648
Epoch 30: 0.8411
Epoch 31: 0.8147
Epoch 32: 0.8599
Epoch 33: 0.8490
Epoch 34: 0.8162
Epoch 35: 0.8312
Epoch 36: 0.8297
Epoch 37: 0.8920
Epoch 38: 0.8734
Epoch 39: 0.8742
Epoch 40: 0.8188
Epoch 41: 0.7998
Epoch 42: 0.8017
Epoch 43: 0.7920
Epoch 44: 0.7882
Epoch 45: 0.7995
Epoch 46: 0.8103
Epoch 47: 0.7786
Epoch 48: 0.7671
Epoch 49: 0.7763
Epoch 50: 0.7649
Epoch 51: 0.7706
Epoch 52: 0.8047
Epoch 53: 0.8129
Epoch 54: 0.7306
Epoch 55: 0.7204
Epoch 56: 0.7189
Epoch 57: 0.7150
Epoch 58: 0.7282
Epoch 59: 0.7033
Epoch 

In [13]:
def match_report_topk(net, report_seqs, track_seqs, sub_tracks, top_k=5):
    net.eval()

    with torch.no_grad():
        E_t = net(torch.tensor(track_seqs).float())
        E_r = net(torch.tensor(report_seqs).float())

        dist = torch.cdist(E_r, E_t)  # 🔥 report → track

    results = {}

    for rid in range(len(report_seqs)):
        row = dist[rid].cpu().numpy()
        idxs = np.argsort(row)[:top_k]

        results[rid] = [
            {
                "track_id": sub_tracks[i]["track_id"],
                "score": float(row[i])
            }
            for i in idxs
        ]

    return results

In [14]:
def plot_report_vs_tracks(sensor_df, event_df, topk_results, rid=0):
    fig = go.Figure()

    # REPORT
    row = event_df.iloc[rid]
    lons = safe_parse_array(row["coors.longitudes"])
    lats = safe_parse_array(row["coors.latitudes"])

    fig.add_trace(go.Scatter(
        x=lons,
        y=lats,
        mode="lines+markers",
        name="Report",
        line=dict(width=4)
    ))

    # TRACKS
    for item in topk_results[rid]:
        tid = item["track_id"]

        df = sensor_df[sensor_df["trackId"] == tid].sort_values("timestamp")

        fig.add_trace(go.Scatter(
            x=df["longitude"],
            y=df["latitude"],
            mode="lines",
            name=f"Track {tid}"
        ))

    fig.update_layout(
        title=f"Report {rid} vs Top Tracks",
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        height=600
    )

    return fig

In [18]:
def match_report_topk_unique(net, report_seqs, track_seqs, sub_tracks, top_k=5):
    net.eval()

    with torch.no_grad():
        E_t = net(torch.tensor(track_seqs, dtype=torch.float32))
        E_r = net(torch.tensor(report_seqs, dtype=torch.float32))

        dist = torch.cdist(E_r, E_t)

    results = {}

    for rid in range(len(report_seqs)):
        row = dist[rid].cpu().numpy()

        # 🔥 group by track_id → lấy best score
        track_best = {}

        for i, d in enumerate(row):
            tid = sub_tracks[i]["track_id"]

            if tid not in track_best:
                track_best[tid] = d
            else:
                track_best[tid] = min(track_best[tid], d)  # 🔥 lấy subtrack gần nhất

        # sort theo distance
        sorted_tracks = sorted(track_best.items(), key=lambda x: x[1])

        results[rid] = [
            {"track_id": tid, "score": float(score)}
            for tid, score in sorted_tracks[:top_k]
        ]

    return results

In [19]:
topk_results = match_report_topk_unique(net, report_seqs, track_seqs, sub_tracks, top_k=5)

In [23]:
fig = plot_report_vs_tracks(sensor_df, event_df, topk_results, rid=13)
fig.show()

In [17]:
event_df.iloc[4]

eventMention                                       Aircraft spotted near zone 4
eventType                                                          SURVEILLANCE
obj                                                                Aircraft_358
entityId                                   9d250ade-2a75-467f-895f-2beb8f022f0c
objType                                                                  BOMBER
nation                                                                       RU
quantity                                                                      3
locIds                                     1340acb0-3fc9-41d8-b1e6-21bba9a90ad9
loc                                                                     Zone_70
coors.type                                                           LineString
coors.properties.basedName                                               Base_7
coors.geometry.type                                                  LineString
coors.longitudes              [103.16365